# Big Data com Google Colab
## Semana 2 — CSV, JSON e Parquet

**Duração:** 3 horas  
**Conhecimento prévio:** conteúdo da Semana 1 e Python introdutório  
**Caso utilizado:** registros de vendas de um comércio eletrônico

Nesta aula, os arquivos ficam fora do notebook. O aluno praticará o fluxo completo: selecionar arquivos no computador, enviá-los ao Colab, abri-los com Python, comparar formatos, gerar Parquet e baixar o resultado.


## 1. Tema da aula

Dados precisam ser representados em algum formato para serem armazenados e transportados. O mesmo conjunto de vendas pode ser salvo como CSV, JSON ou Parquet, mas cada formato possui características diferentes.

- **CSV** organiza valores em linhas e colunas separados por um delimitador.
- **JSON** representa dados usando nomes e valores e permite estruturas mais flexíveis.
- **Parquet** é um formato binário e colunar, criado para armazenamento e análise eficiente.

A escolha do formato afeta:

- tamanho do arquivo;
- preservação dos tipos;
- facilidade de leitura por pessoas;
- velocidade de leitura;
- possibilidade de selecionar somente algumas colunas;
- integração com ferramentas de dados.

Formato não é apenas uma extensão no nome do arquivo. Ele determina como os dados são organizados internamente.


### Comparação inicial

| Característica | CSV | JSON | Parquet |
|---|---|---|---|
| Legível como texto | Sim | Sim | Não |
| Guarda nomes dos campos | No cabeçalho | Em cada objeto | No schema |
| Suporta estruturas aninhadas | Não diretamente | Sim | Sim |
| Preserva tipos | Limitado | Parcial | Sim |
| Organização principal | Linhas | Objetos | Colunas |
| Uso comum | Troca simples de tabelas | APIs e eventos | Data lakes e análise |

```mermaid
flowchart LR
    A[Arquivos no computador] --> B[Upload para o Colab]
    B --> C[Leitura com Pandas]
    C --> D[Comparação de CSV e JSON]
    D --> E[Conversão para Parquet]
    E --> F[Análise por colunas]
    F --> G[Download do resultado]
```


## 2. Objetivos de aprendizagem

Ao final desta aula, você deverá conseguir:

1. explicar as diferenças entre CSV, JSON e Parquet;
2. enviar arquivos do computador para o Google Colab;
3. verificar quais arquivos existem na sessão;
4. abrir CSV e JSON com Pandas;
5. interpretar linhas, colunas e tipos;
6. comparar tamanho e tempo de leitura;
7. converter um DataFrame para Parquet;
8. ler apenas colunas selecionadas;
9. baixar um arquivo criado no Colab;
10. entender por que os arquivos desaparecem ao encerrar a sessão.


## 3. Arquivos utilizados

Antes da aula, tenha estes dois arquivos no computador:

- `dados_vendas.csv`
- `eventos_vendas.json`

Eles contêm os mesmos 5.000 pedidos e foram salvos em formatos diferentes. Os nomes precisam ser mantidos para que o código os encontre.

> Os arquivos enviados ficam na máquina temporária do Colab. Eles não são publicados na internet e não alteram os originais do computador.


## 4. Atividade — preparando o Python

Nesta aula utilizaremos:

- `pandas`, com o apelido `pd`, para trabalhar com tabelas;
- `os`, para verificar arquivos e seus tamanhos;
- `time`, para medir a duração de uma leitura.

Essas bibliotecas já fazem parte do ambiente do Colab.


In [ ]:
# Importa as bibliotecas utilizadas durante a aula.
import os
import time

import pandas as pd

# display apresenta DataFrames de forma organizada.
from IPython.display import display

print("Versão do Pandas:", pd.__version__)
print("Ambiente preparado.")


## 5. Atividade — enviando arquivos para o Colab

A função `files.upload()` abre uma janela para selecionar arquivos no computador.

Quando a janela aparecer:

1. selecione `dados_vendas.csv`;
2. selecione também `eventos_vendas.json`;
3. aguarde os dois nomes aparecerem abaixo da célula.

`arquivos_enviados` será um dicionário. Ele associa o nome de cada arquivo ao seu conteúdo.


In [ ]:
# Importa a ferramenta de arquivos do Google Colab.
from google.colab import files

# Abre a janela para selecionar os arquivos no computador.
arquivos_enviados = files.upload()

# Mostra os nomes que foram enviados.
print("Arquivos enviados:", list(arquivos_enviados.keys()))


### Verificando o envio

`os.listdir()` devolve uma lista com os itens existentes na pasta atual. Em uma sessão comum do Colab, os arquivos enviados ficam em `/content`.


In [ ]:
# Guarda os nomes esperados em variáveis.
NOME_CSV = "dados_vendas.csv"
NOME_JSON = "eventos_vendas.json"

# Lista os arquivos disponíveis na sessão.
arquivos_na_sessao = os.listdir()
print("Arquivos disponíveis:")
print(arquivos_na_sessao)


### Conferindo os nomes

Uma condição iniciada por `if` executa um bloco somente quando algo é verdadeiro. `os.path.exists()` verifica se um caminho existe.

Se um arquivo não for encontrado, `FileNotFoundError` interromperá a execução com uma mensagem clara. Nesse caso, volte à célula de upload e envie o arquivo correto.


In [ ]:
# Verifica o CSV.
if not os.path.exists(NOME_CSV):
    raise FileNotFoundError("Envie o arquivo dados_vendas.csv antes de continuar.")

# Verifica o JSON.
if not os.path.exists(NOME_JSON):
    raise FileNotFoundError("Envie o arquivo eventos_vendas.json antes de continuar.")

print("Os dois arquivos foram encontrados.")


## 6. CSV — valores separados por delimitadores

CSV significa **Comma-Separated Values**, ou valores separados por vírgulas. A primeira linha geralmente contém os nomes das colunas. As demais linhas representam registros.

O CSV é simples e amplamente aceito, mas normalmente não guarda informações completas sobre os tipos. Ao abrir o arquivo, a ferramenta precisa inferir se cada coluna representa texto, número ou data.

### Atividade — olhando o texto original

`open()` abre o arquivo. `readline()` lê uma linha e `close()` fecha o acesso ao arquivo.


In [ ]:
# Abre o CSV como texto usando a codificação UTF-8.
arquivo_csv = open(NOME_CSV, mode="r", encoding="utf-8")

# Lê o cabeçalho e o primeiro pedido.
cabecalho_csv = arquivo_csv.readline()
primeiro_pedido_csv = arquivo_csv.readline()

# Fecha o arquivo depois da leitura.
arquivo_csv.close()

print("Cabeçalho:")
print(cabecalho_csv)
print("Primeiro pedido:")
print(primeiro_pedido_csv)


### Atividade — lendo o CSV com Pandas

`pd.read_csv()` interpreta o arquivo e cria um DataFrame. O argumento é o nome do arquivo.

- `.head()` mostra as primeiras linhas.
- `.shape` informa linhas e colunas.
- `.dtypes` apresenta o tipo inferido para cada coluna.


In [ ]:
# Lê o arquivo CSV e cria um DataFrame.
vendas_csv = pd.read_csv(NOME_CSV)

# Mostra dimensões, amostra e tipos.
print("Linhas e colunas:", vendas_csv.shape)
display(vendas_csv.head())
display(vendas_csv.dtypes.to_frame("tipo_inferido"))


Observe que `data_hora` provavelmente apareceu como `object`, que geralmente representa texto. O valor se parece com uma data para uma pessoa, mas o CSV não registrou explicitamente esse tipo.


## 7. JSON — objetos com nomes e valores

JSON significa **JavaScript Object Notation**. Cada registro possui pares no formato `"nome": valor`. Diferentemente do CSV, os nomes dos campos aparecem em cada objeto.

Nosso arquivo usa **JSON Lines**: cada linha contém um objeto JSON completo. Esse formato facilita o processamento de eventos independentes.

### Atividade — olhando o texto original


In [ ]:
# Abre o JSON como texto.
arquivo_json = open(NOME_JSON, mode="r", encoding="utf-8")

# Lê o primeiro objeto JSON.
primeiro_evento_json = arquivo_json.readline()

# Fecha o arquivo.
arquivo_json.close()

print("Primeiro objeto JSON:")
print(primeiro_evento_json)


### Atividade — lendo JSON Lines com Pandas

`pd.read_json()` abre o JSON. O argumento `lines=True` informa que existe um objeto completo por linha. Usamos `convert_dates=False` para observar os tipos antes da conversão de datas.


In [ ]:
# Lê o arquivo JSON Lines e cria outro DataFrame.
vendas_json = pd.read_json(
    NOME_JSON,
    lines=True,
    convert_dates=False,
)

print("Linhas e colunas:", vendas_json.shape)
display(vendas_json.head())
display(vendas_json.dtypes.to_frame("tipo_inferido"))


## 8. Atividade — verificando se os conteúdos são iguais

Como os arquivos foram criados a partir dos mesmos pedidos, esperamos as mesmas dimensões, colunas e valores.

- `list(vendas_csv.columns)` transforma os nomes das colunas em uma lista.
- `.equals()` verifica se dois DataFrames possuem os mesmos valores e tipos.


In [ ]:
# Compara a quantidade de linhas e colunas.
print("Dimensão do CSV:", vendas_csv.shape)
print("Dimensão do JSON:", vendas_json.shape)

# Compara os nomes das colunas.
colunas_iguais = list(vendas_csv.columns) == list(vendas_json.columns)
print("As colunas estão na mesma ordem?", colunas_iguais)

# Compara todos os valores.
conteudo_igual = vendas_csv.equals(vendas_json)
print("Os DataFrames são iguais?", conteudo_igual)


Se a última resposta for `True`, confirmamos que duas representações diferentes podem carregar o mesmo conteúdo lógico.

Isso não significa que CSV e JSON sejam equivalentes em todos os cenários. JSON aceita estruturas aninhadas e costuma repetir os nomes dos campos, enquanto CSV representa naturalmente uma tabela plana.


## 9. Atividade — corrigindo o tipo da data

`pd.to_datetime()` converte textos para o tipo de data do Pandas. O laço `for` aplica a mesma conversão nos dois DataFrames.

Depois criamos `valor_total`: quantidade multiplicada pelo preço e reduzida pelo desconto percentual.


In [ ]:
# Percorre os dois DataFrames.
for tabela in [vendas_csv, vendas_json]:
    # Converte a coluna de texto para data e hora.
    tabela["data_hora"] = pd.to_datetime(tabela["data_hora"])

    # Calcula o valor antes do desconto.
    valor_bruto = tabela["quantidade"] * tabela["preco_unitario"]

    # Converte o percentual para um número entre 0 e 1.
    desconto = tabela["desconto_pct"] / 100

    # Calcula e arredonda o valor final do pedido.
    tabela["valor_total"] = (valor_bruto * (1 - desconto)).round(2)

display(vendas_csv.head())
print("Novo tipo de data_hora:", vendas_csv["data_hora"].dtype)


## 10. Atividade — comparando o tamanho dos arquivos

`os.path.getsize()` devolve o tamanho em bytes. Dividimos por 1024 para obter kilobytes.


In [ ]:
# Mede os dois arquivos.
tamanho_csv_kb = os.path.getsize(NOME_CSV) / 1024
tamanho_json_kb = os.path.getsize(NOME_JSON) / 1024

# Organiza os resultados em uma tabela.
tamanhos_texto = pd.DataFrame({
    "formato": ["CSV", "JSON Lines"],
    "tamanho_kb": [round(tamanho_csv_kb, 2), round(tamanho_json_kb, 2)],
})

display(tamanhos_texto)


O JSON geralmente ocupa mais espaço neste exemplo porque repete o nome de cada campo em todos os objetos. Essa diferença pode mudar com compressão e com a estrutura dos dados.


## 11. Atividade — medindo o tempo de leitura

`time.perf_counter()` registra um instante com precisão. A diferença entre o instante final e o inicial é o tempo gasto.

Os arquivos desta aula são pequenos. Os tempos podem variar e não constituem um benchmark de produção; servem apenas para ensinar a medição.


In [ ]:
# Mede a leitura do CSV.
inicio_csv = time.perf_counter()
teste_csv = pd.read_csv(NOME_CSV)
tempo_csv = time.perf_counter() - inicio_csv

# Mede a leitura do JSON.
inicio_json = time.perf_counter()
teste_json = pd.read_json(NOME_JSON, lines=True, convert_dates=False)
tempo_json = time.perf_counter() - inicio_json

tempos_texto = pd.DataFrame({
    "formato": ["CSV", "JSON Lines"],
    "tempo_segundos": [tempo_csv, tempo_json],
})

display(tempos_texto)


## 12. Parquet — armazenamento organizado por colunas

CSV e JSON são formatos textuais. Parquet é binário, preserva um schema e organiza os dados por colunas.

Imagine uma consulta que precisa somente de `estado` e `valor_total`. Em um formato colunar, a ferramenta pode evitar a leitura das demais colunas.

```mermaid
flowchart TB
    subgraph Formato por linhas
        A1[Pedido 1: todos os campos]
        A2[Pedido 2: todos os campos]
        A3[Pedido 3: todos os campos]
    end
    subgraph Formato por colunas
        B1[Coluna estado]
        B2[Coluna categoria]
        B3[Coluna valor_total]
    end
```

O Pandas usa uma biblioteca como `pyarrow` para trabalhar com Parquet. Ela já está presente nas sessões atuais do Google Colab.

Caso o Colab informe que não encontrou um mecanismo Parquet, execute uma vez:

```python
!pip install -q pyarrow
```

Essa instalação ocorre somente na sessão temporária do Colab.


In [ ]:
# Instala o suporte a Parquet somente na sessão temporária do Colab.
# O parâmetro -q reduz as mensagens exibidas durante a instalação.
!pip install -q pyarrow


### Atividade — criando o arquivo Parquet

`to_parquet()` grava o DataFrame. `index=False` evita salvar o índice numérico criado automaticamente pelo Pandas.


In [ ]:
# Define o nome do novo arquivo.
NOME_PARQUET = "dados_vendas.parquet"

# Salva os dados do CSV no formato Parquet.
vendas_csv.to_parquet(NOME_PARQUET, index=False)

print("Arquivo criado:", NOME_PARQUET)
print("O arquivo existe?", os.path.exists(NOME_PARQUET))


### Atividade — lendo Parquet e comparando

`pd.read_parquet()` reconstrói o DataFrame. Também mediremos tempo e tamanho.


In [ ]:
# Mede a leitura do Parquet.
inicio_parquet = time.perf_counter()
vendas_parquet = pd.read_parquet(NOME_PARQUET)
tempo_parquet = time.perf_counter() - inicio_parquet

# Mede o tamanho do arquivo.
tamanho_parquet_kb = os.path.getsize(NOME_PARQUET) / 1024

# Compara os três formatos.
comparacao_formatos = pd.DataFrame({
    "formato": ["CSV", "JSON Lines", "Parquet"],
    "tamanho_kb": [tamanho_csv_kb, tamanho_json_kb, tamanho_parquet_kb],
    "tempo_leitura_seg": [tempo_csv, tempo_json, tempo_parquet],
})

comparacao_formatos["tamanho_kb"] = comparacao_formatos["tamanho_kb"].round(2)
comparacao_formatos["tempo_leitura_seg"] = comparacao_formatos["tempo_leitura_seg"].round(6)

display(comparacao_formatos)
display(vendas_parquet.dtypes.to_frame("tipo_preservado"))


Observe especialmente `data_hora`. No Parquet, o tipo de data foi preservado. No CSV e no JSON textual, precisamos realizar a conversão depois da leitura.

Os resultados de tamanho e velocidade dependem dos dados, compressão, máquina e operação. Parquet costuma ser vantajoso para análises, mas não substitui todos os formatos.


## 13. Atividade — lendo somente algumas colunas

No Parquet podemos solicitar somente as colunas necessárias usando `columns=[...]`.


In [ ]:
# Escolhe quatro colunas para uma análise resumida.
colunas_necessarias = ["estado", "categoria", "quantidade", "valor_total"]

# Lê somente essas colunas do arquivo Parquet.
vendas_resumidas = pd.read_parquet(
    NOME_PARQUET,
    columns=colunas_necessarias,
)

print("Colunas carregadas:", list(vendas_resumidas.columns))
print("Linhas e colunas:", vendas_resumidas.shape)
display(vendas_resumidas.head())


### Atividade — produzindo um indicador

Agruparemos por estado e somaremos `valor_total`. O resultado mostra como o arquivo colunar pode alimentar uma análise usando somente as colunas necessárias.


In [ ]:
# Agrupa as linhas por estado e soma o valor total.
vendas_por_estado = (
    vendas_resumidas.groupby("estado")["valor_total"]
                    .sum()
                    .round(2)
                    .sort_values(ascending=False)
)

display(vendas_por_estado.to_frame("valor_total"))


## 14. Atividade — baixando o resultado

Os arquivos da sessão são temporários. `files.download()` envia o Parquet criado pelo Colab para a pasta de downloads do navegador.


In [ ]:
# Inicia o download do arquivo produzido na aula.
files.download(NOME_PARQUET)


## 15. O que acontece quando a sessão termina?

O ambiente do Colab é temporário. Quando a sessão é encerrada ou reiniciada:

- arquivos enviados podem desaparecer;
- arquivos produzidos podem desaparecer;
- variáveis e DataFrames são apagados;
- as células precisam ser executadas novamente.

Por isso, resultados importantes devem ser baixados ou gravados em armazenamento persistente. Nesta aula utilizamos download para que o aluno experimente o ciclo completo sem precisar conectar o Google Drive.


## 16. Síntese da aula

### O que aprendemos sobre dados

- CSV é simples, textual e adequado para tabelas planas.
- JSON é textual, flexível e comum em APIs e eventos.
- JSON Lines armazena um objeto independente por linha.
- Parquet é binário, colunar e preserva tipos.
- O melhor formato depende da maneira como os dados serão produzidos e utilizados.

### O que aprendemos sobre Python

- importar bibliotecas;
- enviar e verificar arquivos;
- abrir arquivos de texto;
- ler CSV e JSON com Pandas;
- consultar dimensões, colunas e tipos;
- usar condições com `if`;
- medir tamanho e tempo;
- converter textos para datas;
- criar novas colunas;
- salvar e ler Parquet;
- selecionar colunas e agrupar valores;
- baixar um arquivo produzido.

### Perguntas para conversar com a turma

1. Por que o JSON tende a ser maior neste conjunto?
2. Por que `data_hora` não foi reconhecida inicialmente como data?
3. Em qual situação você escolheria CSV?
4. Em qual situação você escolheria JSON?
5. Por que Parquet é interessante para um data lake?
6. O menor arquivo será sempre o mais rápido?

### Próxima semana

Na Semana 3, os mesmos princípios de tabela e schema serão usados para introduzir DataFrames distribuídos com PySpark.
